# Bandcamp Artist Discog Scraper
Based on:
- dbeley's [bandcamp-library-scraper.py](https://github.com/dbeley/bandcamp-library-scraper/blob/main/bandcamp-library-scraper.py)
- nahnhh's [webcrawl2_movie details.ipynb](https://github.com/nahnhh/top-movies-2005-2025/blob/main/webcrawl2_movie%20details.ipynb)
- diprog's [tls-client-async](https://github.com/diprog/python-tls-client-async) for better URL fetching...

Artist list used: `slushwave-bandcamp-links.txt` compiled in Slushwave Social Club Discord server.

In [17]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import logging
from rich.logging import RichHandler

FORMAT = '%(message)s'
logging.basicConfig(
    level="INFO", format=FORMAT, datefmt="[%X]", handlers=[RichHandler()]
)
log = logging.getLogger('rich')

BASE_DIR = Path.cwd()
LINKS_FILE = BASE_DIR / 'slushwave-bandcamp-links.txt'
OUTPUT_FILE = BASE_DIR / 'bc-albums-image-links.csv'
IMAGES_DIR = BASE_DIR / 'images'

### Use `tls-client` instead of plain requests
TLS fingerprinting gives a more browser-like behavior to bypass anti-bot protections.

In [18]:
from headers import HEADERS_POOL
from bs4 import BeautifulSoup
from async_tls_client import AsyncSession
import asyncio
import os
import random

In [19]:
class BrowserSession:
	def __init__(self):
		self.requests_made = 0
		self.rotate_headers()
		self.retire_after = random.randint(40, 100)

	def rotate_headers(self, headers=None):
		self.session = AsyncSession(
			client_identifier='chrome_120',
			random_tls_extension_order=True
		)
		self.session.proxies.update({'http': os.getenv('mobileproxyuk'), 'https': os.getenv('mobileproxyuk')})
		if headers:
			self.session.headers.update(headers)

	async def get(self, url, **kwargs):
		if self.requests_made >= self.retire_after:
			self.rotate_headers()
			self.requests_made = 0
			self.retire_after = random.randint(40, 100)

		resp = await self.session.get(url,**kwargs)
		self.requests_made += 1
		return resp

In [20]:
import re
def nozero(text):
	return re.sub(r'[\u200b\u200c\u200d\ufeff]', '', text)

In [21]:
url1 = 'https://daysofblue.bandcamp.com/album/--12'
url2 = 'https://noproblematapes.bandcamp.com/album/--89'
url3 = 'https://geometriclullaby.bandcamp.com/album/geo-c07'
random.seed(42)
s = BrowserSession()
soups = []
for url in [url1, url2, url3]:
	r = await s.get(url)
	soup = BeautifulSoup(r.text, 'lxml')
	soups.append(soup)

In [22]:
import json
album_schema = json.loads(soups[0].select("script[type='application/ld+json']")[0].get_text(strip=True))
tralbum_tag = soups[0].select_one("[data-tralbum]")

[nozero(album_schema['name']), # album name
 nozero(album_schema['byArtist']['name']), # artist name
 album_schema['numTracks'], # number of tracks
 album_schema['keywords'] # tags
]

IndexError: list index out of range

In [ ]:
# Get all track urls
import isodate
import pandas as pd
df = pd.DataFrame([
	{
		"url": t['item']['@id']
	}
	for t in album_schema["track"]["itemListElement"]
])
print(df)

                                          url
0  https://daysofblue.bandcamp.com/track/--59
1  https://daysofblue.bandcamp.com/track/--60
2  https://daysofblue.bandcamp.com/track/--61
3  https://daysofblue.bandcamp.com/track/--62
4  https://daysofblue.bandcamp.com/track/--63
5  https://daysofblue.bandcamp.com/track/--64
6  https://daysofblue.bandcamp.com/track/--65
7  https://daysofblue.bandcamp.com/track/--66


#### Get unique track arts
This is how bandcamp art ID works:
1. Bandcamp generates a unique art_ID for each time you upload a track art.
2. The rest uses art_ID from the album art..

We can drop duplicates based on image hash, with some cases we have to delete manually (dobs album).

In [ ]:
import re
import hashlib

async def get_art_id(url):
	session = BrowserSession()
	r = await session.get(url)
	soup = BeautifulSoup(r.text or "", "lxml")

	icon_url = soup.select_one('link[rel="shortcut icon"]')['href']

	img = await session.get(icon_url)
	img_hash = hashlib.sha256(img.content).hexdigest()

	art_id = re.search(r'a(\d+)_', icon_url).group(1)

	return art_id, img_hash


results = await asyncio.gather(
    *(get_art_id(url) for url in df["url"])
)

art_df = pd.DataFrame(
    results,
    columns=["art_id", "img_hash"]
)

print(art_df)

       art_id                                           img_hash
0  0673663423  0313f4ae7f7665db78c8600f43b12d049339c31415c317...
1  1925835267  7026e3a399375550f866f6ab346e5f1862ed79f57eb8fd...
2  1154539314  7026e3a399375550f866f6ab346e5f1862ed79f57eb8fd...
3  0647675875  4e4333ccc9b50286295d9a89e3f15b093e734f9338c49f...
4  0647675875  4e4333ccc9b50286295d9a89e3f15b093e734f9338c49f...
5  0647675875  4e4333ccc9b50286295d9a89e3f15b093e734f9338c49f...
6  0647675875  4e4333ccc9b50286295d9a89e3f15b093e734f9338c49f...
7  0647675875  4e4333ccc9b50286295d9a89e3f15b093e734f9338c49f...


In [ ]:
print(f"Total unique art IDs: {art_df['art_id'].nunique()}, Unique Image Hashes: {art_df['img_hash'].nunique()}")

Total unique art IDs: 4, Unique Image Hashes: 3


#### Get all dates of the album

In [ ]:
tralbum = json.loads(tralbum_tag["data-tralbum"])['current']
tralbum2 = json.loads(tralbum_tag["data-tralbum"])['trackinfo']

In [ ]:
[nozero(tralbum['title']), # album name
 tralbum['art_id'], # album image id: 2569838638 in https://f4.bcbits.com/img/a2569838638_10.jpg
 tralbum['new_date'], # date first created draft
 tralbum['mod_date'], # date last modified
 tralbum['publish_date'], # publication date
 tralbum['release_date'] # release date
 ]

['ලෝක දෙකක් අතර',
 647675875,
 '15 Aug 2025 07:02:46 GMT',
 '13 Feb 2026 15:15:55 GMT',
 '26 Sep 2025 16:02:30 GMT',
 '26 Sep 2025 00:00:00 GMT']

In [ ]:
# Get all track durations
import pandas as pd
track_df = pd.DataFrame([
	{
		"position": t['track_num'],
		"duration": t['duration'],
	}
	for t in tralbum2
])
print(track_df)

   position  duration
0         1  1440.000
1         2   564.789
2         3   274.286
3         4   164.571
4         5   591.000
5         6   497.090
6         7   325.714
7         8   495.433


#### Adding up total runtime of the album:

In [ ]:
from datetime import timedelta
print(timedelta(seconds=int(track_df["duration"].sum())))

1:12:32
